# 📖 딥러닝 Notebook
**3주차 Pytorch 다루는 법 요약**

이 노트북은 Gemini를 통해 만든 3주차 수업의 핵심 개념을 직접 실행하며 학습할 수 있는 요약본입니다. 

- **실습 방법**: 각 코드 셀을 순서대로 실행(`Shift` + `Enter`)하며 출력되는 텐서의 차원(Shape)과 값을 확인하세요.
- 용어 요약은 **최하단**에 있습니다.
  

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# GPU 가속 활성화 여부
print('PyTorch version:', torch.__version__)
print('GPU available  :', torch.cuda.is_available())
# Visual Studio Code의 "Colab" 확장을 사용하면 Colab의 GPU 가속 사용 가능

PyTorch version: 2.10.0+cu128
GPU available  : True


---
## 2. 이론적/수학적 배경

[Image of neural network forward and backward pass]

### 2.1 선형 변환 (Linear Transformation)
신경망의 기본 레이어인 `nn.Linear`는 입력 행렬 $X$에 대해 가중치(Weight) $W$와 편향(Bias) $b$를 연산하여 특징 공간을 투영합니다.
$$y = XW^T + b$$

### 2.2 손실 함수 (Loss Function)
예측값 $\hat{y}$와 실제 정답 $y$ 사이의 오차 거리를 수치화하여 모델의 현재 상태를 평가합니다.
* **회귀 (MSE Loss):** $L = \frac{1}{N}\sum_{i=1}^N (y_i - \hat{y}_i)^2$
* **분류 (BCE Loss):** $L = -\frac{1}{N}\sum_{i=1}^N [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$

### 2.3 최적화 (Optimization)와 역전파 (Backpropagation)
손실 값 $L$에 대해 각 파라미터가 미치는 영향을 연쇄 법칙(Chain Rule)으로 계산하여 기울기(Gradient)를 구한 뒤, 학습률 $\eta$에 따라 파라미터를 업데이트합니다.
$$W \leftarrow W - \eta \frac{\partial L}{\partial W}$$
---

In [ ]:
# 1. 데이터 준비 (Dataset & DataLoader)
X = torch.randn(100, 10) 
y = torch.randint(0, 2, (100, 1)).float()  

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True) 

# 2. 모델 설계 (nn.Module 기반 커스텀 모델)
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 1), # 10개의 입력을 받아 1개의 출력을 내보냄
            nn.Sigmoid() # 그 1개의 출력에 sigmoid를 취함
        )
        
    def forward(self, x):
        return self.network(x) # (Batch, 10) -> (Batch, 1)

model = MyModel()

# 3. 손실 함수 및 옵티마이저 설정
criterion = nn.BCELoss() # 이진 분류용 오차 계산기
optimizer = optim.SGD(model.parameters(), lr=0.01) # SGD는 기울기(gradient)를 이용해 가중치를 업데이트하는 최적화 알고리즘의 일종.

In [ ]:
# 4. 표준 5단계 학습 루프
epochs = 5
for epoch in range(epochs):
    for batch_X, batch_y in dataloader:
        
        # [1단계] 준비: 이전 오답 기록 지우기
        optimizer.zero_grad() 
        
        # [2단계] 예측: 문제 풀고 가짜 답 도출 (Forward)
        y_pred = model(batch_X) 
        
        # [3단계] 채점: 정답과 비교해 오차 계산
        loss = criterion(y_pred, batch_y) 
        
        # [4단계] 분석: 왜 틀렸는지 기울기 구하기 (Backward)
        loss.backward() 
        
        # [5단계] 수정: 가중치를 실제로 고치기 (Step)
        optimizer.step() 
        
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')

---
## 3. PyTorch를 통한 자동화 (Forward & Backward)

지난주까지 여러분은 오차를 줄이기 위해 가중치 $W$를 어떻게 업데이트할지, 연쇄 법칙(Chain Rule)을 적용해 손으로 직접 편미분을 계산했습니다. 하지만 실무에서 수백 개의 층을 가진 모델을 손으로 미분하는 것은 불가능합니다. 파이토치는 이 과정을 **순전파(Forward)**와 **역전파(Backward)**라는 두 단계로 완벽하게 자동화합니다.



### ➡️ 순전파 (Forward Pass): "일단 풀어봐!"
입력 데이터 $x$가 모델의 여러 레이어를 통과하며 최종 예측값 $\hat{y}$를 계산해내는 과정입니다. 이때 파이토치는 단순히 계산만 하는 것이 아니라, **나중에 미분하기 위해 데이터가 지나간 길(연산 그래프)을 전부 기록**해 둡니다.
$$\hat{y} = \sigma(Wx + b)$$

### ⬅️ 역전파 (Backward Pass): "어디서 틀렸는지 거꾸로 추적해!"
예측값 $\hat{y}$와 정답 $y$를 비교해 오차(Loss)가 나오면, 파이토치의 `Autograd`(자동 미분) 엔진이 작동합니다. 최종 오차에서부터 거꾸로 되돌아가며, 순전파 때 기록해둔 길을 따라 **각 가중치가 오차에 얼마나 기여했는지(기울기, Gradient)를 자동으로 계산**합니다.
$$\frac{\partial L}{\partial W}, \quad \frac{\partial L}{\partial b}$$

In [ ]:
import torch

# 1. 텐서 생성 시 requires_grad = True를 켜주면, 
# "이 변수가 참여하는 모든 연산을 기록해서 나중에 미분해줘!"라는 뜻입니다.
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# 2. 순전파 (Forward)
# 계산 과정이 내부적으로 그래프 형태로 기록됩니다.
y_pred = w * x + b   # 예측값: 3*2 + 1 = 7
loss = (y_pred - 10)**2 # 정답이 10이라고 가정할 때의 오차 (MSE)

print(f"순전파 직후 w의 기울기: {w.grad}") # 아직 역전파 전이므로 None

# 3. 역전파 (Backward)
# 손으로 하던 편미분 연쇄 법칙을 단 한 줄로 끝냅니다.
loss.backward()

# loss를 w로 편미분한 값이 w.grad에 자동으로 저장됩니다.
print(f"역전파 이후 w의 기울기: {w.grad}") # 결과가 자동으로 계산되어 들어있음!
# 이후 epoch 반복을 시킬 땐 매 미니배치마다 optimizer.zero_grad()로 기울기를 다시 초기화해주어야 합니다.

### 학습 흐름(Forward/Backward) 핵심 용어

| 분류 | 용어 / 코드 | 기술적 정의 및 역할 |
| :--- | :--- | :--- |
| **미분 추적** | `requires_grad=True` | 텐서를 생성할 때, 해당 텐서에 대한 연산 기록을 남겨 미분값을 계산할 준비를 함 |
| | `Autograd` | 파이토치의 심장. 텐서들의 모든 연산 과정을 추적하여 연쇄 법칙(미분)을 자동 수행하는 엔진 |
| **순방향 연산** | `Forward Pass` | 입력 데이터가 신경망의 가중치들과 곱해지고 더해져서 최종 예측값(출력)을 만들어내는 과정 |
| **역방향 미분** | `loss.backward()` | 계산된 최종 오차(Loss)에서부터 출발해, 역방향으로 모든 파라미터의 편미분(기울기) 값을 자동 계산하여 `.grad` 속성에 저장 |
| **파라미터** | `w.grad` | `backward()` 실행 후, 해당 가중치(`w`)를 얼마나 수정해야 할지 알려주는 기울기(미분) 값이 담기는 공간 |

---
## 4. 최적화와 용어들

* **메모리 최적화 (`with torch.no_grad():`)**: 검증(Validation) 및 테스트 단계에서는 모델을 업데이트할 필요가 없습니다. 
  블록을 씌워 Autograd 엔진을 비활성화하면 연산 속도가 향상되고 GPU 메모리 사용량을 크게 줄일 수 있습니다.
* **메모리 누수 방지 (`.item()`)**: 학습 루프 내에서 로그를 출력하거나 loss 값을 누적할 때, `loss` 텐서 자체를 더하면
   연산 그래프가 통째로 메모리에 누적됩니다. 순수 파이썬 스칼라 값만 추출하는 `.item()`을 반드시 사용하세요.
* **차원 조작 (`view` vs `reshape`)**: 차원 불일치 에러 대응 시, `c.view(2, -1)`처럼 `-1`을 활용하면 
  배치 사이즈 변동에 유연하게 대응할 수 있습니다.

---
## 5. 핵심 용어 및 코드 컴팩트 요약

### 5.1 텐서 및 데이터 파이프라인
| 분류 | 용어 / 코드 | 기술적 정의 및 역할 |
| :--- | :--- | :--- |
| **자료형** | `Tensor` / `torch.tensor([])` | GPU 연산 및 자동 미분을 지원하는 파이토치 기본 다차원 배열 |
| | `from_numpy(arr)` | 메모리를 공유하여 기존 NumPy 배열을 Tensor로 변환 |
| | `.view()` / `.reshape()` | 데이터 원소 수는 유지하되 텐서의 차원/형태 재배치 |
| **데이터 공급** | `Dataset` / `TensorDataset(X, y)` | 입력 데이터(X)와 정답 라벨(y)을 같은 인덱스 순서로 묶어 관리 |
| | `DataLoader(..., batch_size, shuffle)` | 데이터를 묶고(Batch) 무작위로 섞어(Shuffle) 모델에 순차 공급 |

### 5.2 모델 구성 및 자동 미분 (Autograd)
| 분류 | 용어 / 코드 | 기술적 정의 및 역할 |
| :--- | :--- | :--- |
| **자동 미분** | `Autograd` / `requires_grad=True` | 텐서의 연산 과정을 기록하여 자동 미분(기울기 계산) 스위치 ON |
| | `with torch.no_grad():` | 평가/테스트 시 미분 기록을 꺼서 메모리와 속도 최적화 |
| | `.item()` | 원소가 1개인 텐서에서 순수 파이썬 숫자(실수)만 추출 (메모리 누수 방지) |
| **모델 구성** | `nn.Linear(in, out)` | $XW^T+b$ 연산을 수행하는 선형 회귀 레이어 (가중치/편향 자동 생성) |
| | `nn.Sequential(L1, L2)` | 여러 레이어를 묶어 순차적으로 데이터가 통과하게 하는 컨테이너 |
| | `nn.Module` | 신경망 구조를 직접 설계할 때 상속받는 기본 뼈대 베이스 클래스 |

### 5.3 학습 루프 5단계 핵심 로직
| 단계 | 코드 | 기술적 정의 및 역할 |
| :--- | :--- | :--- |
| **1. 준비** | `optimizer.zero_grad()` | 이전 반복에서 누적된 기울기(미분값)을 0으로 리셋 |
| **2. 예측** | `y_pred = model(X)` | 데이터를 모델에 통과(Forward)시켜 현재 가중치 기반 가짜 답 도출 |
| **3. 채점** | `loss = criterion(y_pred, y)` | 손실 함수(MSE, BCE 등)로 예측값과 정답의 차이를 오차 점수로 수치화 |
| **4. 분석** | `loss.backward()` | 역전파 수행, 연쇄 법칙으로 각 파라미터가 오차에 미친 기여도(기울기) 산출 |
| **5. 수정** | `optimizer.step()` | 계산된 기울기를 바탕으로 옵티마이저가 실제 가중치를 정답에 가깝게 업데이트 |

### 5.4 기울기 초기화 관련 핵심 용어

| 분류 | 코드 | 기술적 정의 및 역할 |
| :--- | :--- | :--- |
| **기울기 초기화** | `optimizer.zero_grad()` | 옵티마이저에 연결된 모든 파라미터의 `.grad` (기울기 값)를 `0`으로 리셋 |
| | `model.zero_grad()` | 모델 전체 파라미터의 기울기를 0으로 만듦 (일반적으로 `optimizer.zero_grad()`와 동일한 효과이나, 옵티마이저 기준 호출이 표준) |
| **작동 원리** | `Gradient Accumulation` | 파이토치가 역전파(`backward`) 시 미분값을 덮어쓰지 않고 계속 더하는(누적) 특성 |

---
## 번외1) 모델 선언의 정석 (`nn.Sequential` 블록 쌓기)

파이토치에서 가장 기초적이고 직관적인 모델 선언 방식은 `nn.Sequential`을 사용하는 것입니다. 이는 데이터가 통과할 파이프라인을 순서대로 조립하는 것과 같습니다. 

**기본 공식: `Linear` $\rightarrow$ `Activation` $\rightarrow$ `Linear`**

선형 레이어(`nn.Linear`)만 연속으로 쌓으면 수학적으로 아무리 깊게 쌓아도 결국 하나의 층과 똑같아집니다. ($W_2(W_1x) = (W_2W_1)x = W_{new}x$). 따라서 층과 층 사이에는 반드시 데이터를 '구부려주는' 비선형 활성화 함수(Activation Function)가 들어가야 합니다.

In [3]:
import torch.nn as nn

# [예시] 입력층(특성 4개) -> 은닉층(8개) -> 출력층(1개)의 구조
simple_model = nn.Sequential(
    nn.Linear(4, 8),    # 1. 4개의 데이터를 받아 8개로 늘려 특징을 추출
    nn.Sigmoid(),       # 2. 선형 계산 결과를 0~1 사이로 구부려줌 (비선형성 추가)
    nn.Linear(8, 1)     # 3. 8개의 특징을 모아 최종 1개의 결과 도출
)

print(simple_model)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=8, out_features=1, bias=True)
)


---
## 번외2) Sigmoid 말고 또 뭐가 들어갈 수 있을까? (활성화 함수)



[Image of neural network activation functions]


은닉층(Hidden Layer)에서 레이어 사이에 들어가는 함수들은 모델의 학습 능력을 결정합니다. 3주차에서 반드시 알아야 할 두 가지는 다음과 같습니다.

* **`nn.Sigmoid()`:** * 계산된 값을 0과 1 사이로 압축합니다. 
  * 초기 딥러닝에서 많이 썼지만, 층이 깊어지면 미분값이 0에 가까워져 학습이 멈추는 **기울기 소실(Vanishing Gradient)** 문제가 발생합니다. 현재는 주로 '이진 분류의 최종 출력용'으로만 씁니다.
* **`nn.ReLU()` (Rectified Linear Unit):** * $f(x) = \max(0, x)$
  * 음수는 0으로 버리고, 양수는 그대로 통과시킵니다. 계산이 매우 빠르고 기울기 소실 문제를 해결하여 **현대 딥러닝 은닉층의 기본(Default)**으로 사용됩니다.

**요즘 스타일의 모델 선언:**
`Linear` $\rightarrow$ `ReLU` $\rightarrow$ `Linear` $\rightarrow$ `ReLU` ... 형태로 쌓습니다.

---
## 번외3) 손실 함수(Loss)와 출력층의 짝꿍

모델이 어떤 문제를 푸느냐에 따라 **마지막 레이어의 형태**와 **손실 함수(Loss Function)**는 반드시 세트로 맞춰주어야 합니다. 이 조합이 틀리면 학습이 전혀 되지 않습니다.

### 🎯 케이스 1: 연속적인 수치 예측 (회귀 - Regression)
* **목표:** 집값, 주식 가격, 온도 등 구체적인 숫자를 맞추기
* **마지막 레이어:** 아무런 함수도 붙이지 않고 `nn.Linear`로 끝냅니다. (어떤 숫자든 나와야 하므로)
* **손실 함수:** **`nn.MSELoss()`** (Mean Squared Error). 정답과 예측값의 차이를 제곱하여 오차를 계산합니다.
  * $Loss = \frac{1}{N}\sum (y - \hat{y})^2$

### 🎯 케이스 2: O / X 맞추기 (이진 분류 - Binary Classification)
* **목표:** 합격/불합격, 스팸/정상, 양성/음성 등 2가지 중 하나로 분류하기
* **마지막 레이어:** 결과를 0% ~ 100% (0~1) 확률로 나타내야 하므로 **`nn.Sigmoid()`**로 끝냅니다.
* **손실 함수:** **`nn.BCELoss()`** (Binary Cross Entropy). 확률 기반으로 정답과의 차이를 극대화하여 계산합니다.
  * $Loss = -[y \log(\hat{y}) + (1-y)\log(1-\hat{y})]$